# Week 2 — Manual Attention Implementation

**Goal:** Build a scaled dot-product attention forward pass from scratch using `torch.einsum`, with no TransformerLens. Understand what each line does mathematically before running it.

**By the end of this notebook you should be able to:**
- Explain what Q, K, V are and where they come from
- Derive the attention score formula $QK^T / \sqrt{d_{\text{head}}}$ step by step
- Implement multi-head attention manually and verify tensor shapes at each step
- Match your output against TransformerLens as a sanity check

**Question to sit with:** The formula $QK^T / \sqrt{d_{\text{head}}}$ has three parts — the dot product, the transpose, and the scaling. Why does each one exist? What breaks if you remove it?

## Section 1 — Setup and toy dimensions

We'll use a tiny model to keep shapes readable:
- `n_tokens = 3` (sequence length)
- `d_model = 8` (residual stream width)
- `n_heads = 2`
- `d_head = 4` (so d_model = n_heads × d_head)

In [1]:
import torch
import einops
from einops import einsum as esum
import math

torch.manual_seed(42)

# Toy dimensions
n_tokens = 3
d_model = 8
n_heads = 2
d_head = d_model // n_heads  # = 4

print(f"n_tokens={n_tokens}, d_model={d_model}, n_heads={n_heads}, d_head={d_head}")

# A fake residual stream: [n_tokens, d_model]
x = torch.randn(n_tokens, d_model)
print(f"\nResidual stream x: {x.shape}")

n_tokens=3, d_model=8, n_heads=2, d_head=4

Residual stream x: torch.Size([3, 8])


## Section 2 — Weight matrices

Each attention head has its own Q, K, V projection matrices. These project from `d_model` down to `d_head`.

We'll store them stacked: `W_Q` has shape `[n_heads, d_model, d_head]` — one projection matrix per head.

In [2]:
# Weight matrices: [n_heads, d_model, d_head]
W_Q = torch.randn(n_heads, d_model, d_head) * 0.02
W_K = torch.randn(n_heads, d_model, d_head) * 0.02
W_V = torch.randn(n_heads, d_model, d_head) * 0.02
W_O = torch.randn(n_heads, d_head, d_model) * 0.02  # output projection: d_head -> d_model

print(f"W_Q: {W_Q.shape}  (n_heads, d_model, d_head)")
print(f"W_K: {W_K.shape}")
print(f"W_V: {W_V.shape}")
print(f"W_O: {W_O.shape}  (n_heads, d_head, d_model)")

W_Q: torch.Size([2, 8, 4])  (n_heads, d_model, d_head)
W_K: torch.Size([2, 8, 4])
W_V: torch.Size([2, 8, 4])
W_O: torch.Size([2, 4, 8])  (n_heads, d_head, d_model)


## Section 3 — Compute Q, K, V

Project the residual stream through each weight matrix to get queries, keys, and values.

`Q[head, token, d_head] = x[token, d_model] @ W_Q[head, d_model, d_head]`

In einsum notation: `"seq d_model, n_heads d_model d_head -> n_heads seq d_head"`

In [3]:
# Project residual stream into Q, K, V for each head
Q = torch.einsum("s m, h m d -> h s d", x, W_Q)  # [n_heads, n_tokens, d_head]
K = torch.einsum("s m, h m d -> h s d", x, W_K)
V = torch.einsum("s m, h m d -> h s d", x, W_V)

print(f"Q: {Q.shape}  (n_heads, n_tokens, d_head)")
print(f"K: {K.shape}")
print(f"V: {V.shape}")

# TODO: annotate what each axis represents for Q[0] (head 0)
# Q[0] has shape [n_tokens, d_head] — it's the query matrix for head 0

Q: torch.Size([2, 3, 4])  (n_heads, n_tokens, d_head)
K: torch.Size([2, 3, 4])
V: torch.Size([2, 3, 4])


## Section 4 — Attention scores

The raw attention score between query token $i$ and key token $j$ is the dot product $Q_i \cdot K_j$.

For all pairs at once: `attn_scores[head, query_pos, key_pos] = Q[head, query_pos] · K[head, key_pos]`

Then divide by $\sqrt{d_{\text{head}}}$ to stabilize gradients.

**Why the scaling?** Without it, dot products grow with $d_{\text{head}}$, pushing softmax into a nearly-one-hot regime where gradients vanish. Dividing by $\sqrt{d_{\text{head}}}$ keeps the variance of the scores roughly constant regardless of head size.

In [4]:
# Attention scores: [n_heads, n_tokens, n_tokens]
# attn_scores[h, i, j] = dot(Q[h,i], K[h,j]) / sqrt(d_head)
attn_scores = torch.einsum("h i d, h j d -> h i j", Q, K) / math.sqrt(d_head)

print(f"attn_scores: {attn_scores.shape}  (n_heads, query_pos, key_pos)")
print(f"\nHead 0 scores (query × key):\n{attn_scores[0].round(decimals=3)}")

# Causal mask: token i can only attend to tokens j <= i
# (comment this out if you want to see full bidirectional attention)
mask = torch.triu(torch.ones(n_tokens, n_tokens), diagonal=1).bool()
attn_scores_masked = attn_scores.masked_fill(mask, float('-inf'))
print(f"\nHead 0 after causal mask:\n{attn_scores_masked[0].round(decimals=3)}")

attn_scores: torch.Size([2, 3, 3])  (n_heads, query_pos, key_pos)

Head 0 scores (query × key):
tensor([[-0.0080,  0.0010, -0.0010],
        [-0.0040,  0.0000, -0.0010],
        [-0.0020,  0.0000, -0.0000]])

Head 0 after causal mask:
tensor([[-0.0080,    -inf,    -inf],
        [-0.0040,  0.0000,    -inf],
        [-0.0020,  0.0000, -0.0000]])


## Section 5 — Softmax and weighted sum over values

Softmax converts raw scores into a probability distribution over key positions — each row sums to 1.

Then we take a weighted sum of value vectors: `z[head, query_pos] = sum_j( attn_pattern[head, query_pos, j] * V[head, j] )`

In [5]:
# Softmax over key positions (last dim)
attn_pattern = torch.softmax(attn_scores_masked, dim=-1)  # [n_heads, n_tokens, n_tokens]

print(f"attn_pattern: {attn_pattern.shape}")
print(f"\nHead 0 attention pattern (rows sum to 1):\n{attn_pattern[0].round(decimals=3)}")
print(f"\nRow sums (should all be 1.0): {attn_pattern[0].sum(dim=-1).round(decimals=3)}")

# Weighted sum over value vectors
z = torch.einsum("h i j, h j d -> h i d", attn_pattern, V)  # [n_heads, n_tokens, d_head]
print(f"\nz (pre-output projection): {z.shape}  (n_heads, n_tokens, d_head)")

attn_pattern: torch.Size([2, 3, 3])

Head 0 attention pattern (rows sum to 1):
tensor([[1.0000, 0.0000, 0.0000],
        [0.4990, 0.5010, 0.0000],
        [0.3330, 0.3340, 0.3330]])

Row sums (should all be 1.0): tensor([1., 1., 1.])

z (pre-output projection): torch.Size([2, 3, 4])  (n_heads, n_tokens, d_head)


## Section 6 — Output projection and residual stream write

Each head projects its output `z` back to `d_model` via `W_O`, then all heads are summed and added back to the residual stream.

This is the "write" step: each head independently computes a `d_model`-dimensional update and the updates are superimposed.

In [6]:
# Project each head's output back to d_model: [n_heads, n_tokens, d_model]
out_per_head = torch.einsum("h s d, h d m -> h s m", z, W_O)

print(f"out_per_head: {out_per_head.shape}  (n_heads, n_tokens, d_model)")

# Sum over heads and add to residual stream
attn_output = out_per_head.sum(dim=0)  # [n_tokens, d_model]
x_new = x + attn_output               # residual connection

print(f"attn_output: {attn_output.shape}  (n_tokens, d_model)")
print(f"x_new (updated residual stream): {x_new.shape}")

out_per_head: torch.Size([2, 3, 8])  (n_heads, n_tokens, d_model)
attn_output: torch.Size([3, 8])  (n_tokens, d_model)
x_new (updated residual stream): torch.Size([3, 8])


## Section 7 — Einsum practice

Quick exercises to build fluency with einsum notation. Write each operation using `torch.einsum` and verify the output shape.

In [ ]:
a = torch.randn(4)
b = torch.randn(4)
A = torch.randn(3, 4)
B = torch.randn(4, 5)
C = torch.randn(2, 3, 4)
D = torch.randn(2, 4, 5)

# Exercise 1: dot product of two vectors
# inputs: a [4], b [4] — expected output shape: scalar ()
# dot = torch.einsum("???", a, b)
dot = torch.einsum("i, i ->", a, b)
assert dot.shape == torch.Size([]), f"Expected scalar, got {dot.shape}"
print(f"1. Dot product: {dot.item():.4f}")

# Exercise 2: matrix multiply A @ B
# inputs: A [3, 4], B [4, 5] — expected output shape: (3, 5)
# mm = torch.einsum("???", A, B)
mm = torch.einsum("i j, j k -> i k", A, B)
assert mm.shape == (3, 5), f"Expected (3,5), got {mm.shape}"
print(f"2. MatMul shape: {mm.shape}")

# Exercise 3: batched matrix multiply C @ D
# inputs: C [2, 3, 4], D [2, 4, 5] — expected output shape: (2, 3, 5)
# bmm = torch.einsum("???", C, D)
bmm = torch.einsum("b i j, b j k -> b i k", C, D)
assert bmm.shape == (2, 3, 5), f"Expected (2,3,5), got {bmm.shape}"
print(f"3. Batched matmul shape: {bmm.shape}")

# Exercise 4: transpose A
# input: A [3, 4] — expected output shape: (4, 3)
# At = torch.einsum("???", A)
At = torch.einsum("i j -> j i", A)
assert At.shape == (4, 3)
print(f"4. Transpose shape: {At.shape}")

# Exercise 5: outer product of a and b
# inputs: a [4], b [4] — expected output shape: (4, 4)
# outer = torch.einsum("???", a, b)
outer = torch.einsum("i, j -> i j", a, b)
assert outer.shape == (4, 4)
print(f"5. Outer product shape: {outer.shape}")

print("\nAll assertions passed.")

## Section 8 — Reflection

Answer these before moving on:

1. **The transpose:** In `Q @ K^T`, why do we transpose K? What would the shapes be without the transpose?

2. **The scaling:** Remove the `/ math.sqrt(d_head)` and re-run Section 4. What happens to the attention pattern after softmax? Why?

3. **The dot product:** Why is dot product a meaningful measure of "how much should token i attend to token j"? What geometric property does it exploit?

4. **The residual stream:** In Section 6, each head writes independently and the writes are summed. Why does this matter for how we interpret circuits?

*(Write your answers in the cell below.)*

**Your answers:**

1. We transpose K so the shapes work: Q is [n_tokens, d_head] and K is [n_tokens, d_head], so Q @ K gives a shape mismatch. With the transpose, K becomes [d_head, n_tokens], giving [n_tokens, d_head] @ [d_head, n_tokens] = [n_tokens, n_tokens] — a score for every (query, key) pair.

2. Without scaling, dot products grow in magnitude with d_head, pushing softmax toward a one-hot distribution (one token gets ~1.0, rest ~0.0). In that regime softmax has near-zero gradient and the model can't learn. Dividing by √d_head keeps score variance roughly constant.

3. Dot product measures how aligned two vectors are — large when they point in similar directions, zero when perpendicular. The model learns to make query vectors point in directions that match key vectors for relevant tokens. High dot product = "these tokens are relevant to each other."

4. Because heads write independently and sum, you can zero out one head and measure exactly what it contributed. That's what makes circuit analysis possible — if heads were entangled, you couldn't isolate them.